# Gold: Wikipedia Clickstream Top Pages (v1)

Reads the `wikipedia.clickstream_enwiki` bronze table (see
`data-generators/bronze_ingest_wikipedia_clickstream.ipynb`) and aggregates
it by destination page: total incoming clicks and distinct referrer count.
Persists the result as a managed Spark table for demo dashboards/queries.

Re-runnable: `mode("overwrite")`, so re-running just refreshes the table.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

## Configuration

In [ ]:
DATABASE_NAME = "wikipedia"
SOURCE_TABLE = "clickstream_enwiki"
TARGET_TABLE = "gold_clickstream_top_pages_v1"

## Load bronze table

In [ ]:
df = spark.table(f"{DATABASE_NAME}.{SOURCE_TABLE}")

## Transform: top pages by incoming clicks

Aggregates by destination page (`curr`): total clicks (`sum(n)`) and count
of distinct referring pages (`prev`).

In [ ]:
summary_df = (
    df
    .groupBy(F.col("curr").alias("page"))
    .agg(
        F.sum("n").alias("total_clicks"),
        F.countDistinct("prev").alias("distinct_referrers"),
    )
    .orderBy(F.desc("total_clicks"))
)

## Persist as a managed Spark table

In [ ]:
summary_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TARGET_TABLE}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_pages FROM {DATABASE_NAME}.{TARGET_TABLE}").show()
spark.sql(f"SELECT * FROM {DATABASE_NAME}.{TARGET_TABLE} ORDER BY total_clicks DESC LIMIT 10").show()